In diesem Notebook werden **die Ergebnisse (Predictions)** des Event-Classifiers mit den **unterschiedlichen Goldstandards verglichen** **(pro Text)**

1. **Predictions laden (einzeln pro Text) und Labelverteilungen ausgeben**
2. **Alle Goldstandards laden und Labelverteilungen ausgeben** (enthalten keine Spanindizes)
3. **Predictions und Goldstandardspannen matchen** 
    - Dafür wird zunächst "261112_item_mapped_final_data.json" geladen, weil es sowohl die spans als auch die task-ids enthält
    - Gruppieren der Goldspans nach Startindex für späteren Abgleich
    - Spans der Predictions laden
    - Abgleich der Spans nach folgendem Muster: (1) gleicher Start und gleiches Ende, (2) Start-Match und (3) End-Match
    - "relevante" Task-IDs speichern
    - DF für die Goldstandards erstellen "df_gold_comparison", der nur die vergleichbaren items enthält
    - Dictionary von Task-ID und Predictions erstellen und daraufhin, und basierebnd darauf die Labels extrahieren und Spalte mit Predicitionlabels an "df_gold_comparison" hängen
4. **Konfusionsmatrizen erstellen** (absolute Zahlen)
5. **F1-Scores ermitteln** zum allgemeinen Vergleich, dann aber aucgh pro Label

Execute event-classifier:

python3 predict.py plain-text-file \
"/Users/momamari/Codes/event_classifier/demo_model" \
"/Users/momamari/Codes/event_classifier/plaintexts/von_Kleist_Heinrich_Das_Erdbeben_in_Chili.txt" \
"/Users/momamari/Codes/event_classifier/predictions/Erdbeben_predictions.json" \
--device==cpu


In [1]:
import json
from collections import Counter
import pandas as pd


## 1. Predictions laden

In [2]:
prediction_path = "../data/results_classifier/20261117_effineu_predictions.json"

with open(prediction_path) as file:
    predictions = json.load(file)

# # #mit indents speicher
# with open("/Users/momamari/Codes/event_classifier/results/20261116_effi_predictions.json","w") as file:
#     json.dump(predictions, file,indent=4)

#### Tagverteilung ausgeben

In [659]:
### Explore evaluation results

tagverteilung = list()
for k, v in predictions.items():
    if k == "annotations":
        print("N annotations: ",len(v))
        for anno in v:
            tag = anno.get("predicted")
            tagverteilung.append(tag)


label_counts = Counter(tagverteilung)
print(label_counts)


N annotations:  11380
Counter({'process': 4050, 'stative_event': 3781, 'non_event': 3376, 'change_of_state': 173})


## 2. Load all gold truths

In [635]:

all_gold_labels = pd.read_csv("../data/effi/261311_goldlabel_vgl.csv")

### Get label counts per goldstandard

In [636]:
columns = ["label", "ds_groundtruth", "mace_ground_truth", "glad_ground_truth"]

for col in columns:
    print(f"\n{col}:")
    print(all_gold_labels[col].value_counts())


label:
label
non_event          2686
process            1968
stative_event      1578
change_of_state      39
Name: count, dtype: int64

ds_groundtruth:
ds_groundtruth
non_event          2929
process            2351
stative_event       989
change_of_state       2
Name: count, dtype: int64

mace_ground_truth:
mace_ground_truth
process            2756
non_event          2281
stative_event      1230
change_of_state       4
Name: count, dtype: int64

glad_ground_truth:
glad_ground_truth
process            2433
non_event          2385
stative_event      1437
change_of_state      16
Name: count, dtype: int64


## 3. Predictions und Goldstandardspannen matchen

### 3.1. Die Spans und task ids extrahieren

In [3]:
probabilistic_gold = "../data/261112_item_mapped_final_data.json"
with open(probabilistic_gold) as file:
    probabilistic_index_spans = json.load(file)



In [638]:
rows = []

for title, tasks in probabilistic_index_spans.items():
    if title == "Effi Briest":
        for taskid, taskdata in tasks.items():
            span = taskdata["annotations"]["gold_standard"]["span"]

            rows.append({
                "title": title,
                "task": taskid,
                "span": span
            })

df_spans = pd.DataFrame(rows)

In [639]:
df_spans.head()

,title,task,span
0,Effi Briest,1,"[291557, 291648]"
1,Effi Briest,2,"[[319450, 319486]]"
2,Effi Briest,3,"[[333226, 333272]]"
3,Effi Briest,4,"[[287643, 287707]]"
4,Effi Briest,5,"[[297396, 297417]]"


In [640]:
df_spans.head()
print(len(df_spans))

6271


#### Group gold spans by start index
- key is start index, value is "[{'task': '1', 'end': 29783}]"

In [641]:
# Goldstandard-Spans nach Startindex gruppieren
gold_by_start = {}

for _, row in df_spans.iterrows():
    span = row["span"]

    if isinstance(span[0], list):
        start, end = span[0]
    else:
        start, end = span

    gold_by_start.setdefault(start, []).append({
        "task": row["task"],
        "start": start,
        "end": end
    })

#### Predicted Spans extrahieren

In [642]:
predicted_spans = list()
for k, v in predictions.items():
    if k == "annotations":
        print("N annotations: ",len(v))
        for anno in v:
        
            startspan = anno.get("start")
            endspan = anno.get("end")
            predicted_spans.append([startspan,endspan])
            #print(startspan, endspan)

print(predicted_spans)

N annotations:  11380
[[17, 188], [190, 509], [511, 605], [748, 874], [876, 927], [982, 1107], [1109, 1242], [1310, 1317], [1352, 1551], [1568, 1594], [1596, 1838], [1891, 1974], [1976, 2031], [2033, 2117], [2119, 2247], [2350, 2397], [2399, 2452], [2399, 2536], [2473, 2501], [2537, 2551], [2675, 2800], [2802, 3136], [2806, 2830], [2831, 2908], [3002, 3046], [3138, 3211], [3213, 3291], [3302, 3310], [3312, 3372], [3398, 3428], [3430, 3544], [3546, 3573], [3575, 3613], [3615, 3679], [3682, 3793], [3795, 3865], [3874, 3895], [3976, 4038], [4040, 4077], [4045, 4060], [4079, 4099], [4118, 4142], [4146, 4171], [4189, 4246], [4248, 4289], [4308, 4387], [4312, 4341], [4389, 4474], [4393, 4432], [4475, 4484], [4515, 4533], [4572, 4624], [4632, 4660], [4665, 4683], [4690, 4768], [4725, 4750], [4751, 4764], [4768, 4871], [4843, 4864], [4872, 4959], [4970, 5128], [5129, 5200], [5202, 5260], [5261, 5329], [5331, 5363], [5364, 5420], [5473, 5503], [5505, 5547], [5572, 5650], [5653, 5694], [5741, 61

#### Abgleich von prediction und gold spans
-  (1) gleicher Start und gleiches Ende, (2) Start-Match und (3) End-Match

In [ ]:
#prediction spans abgleichen
matches = []
matched_tasks = set()

# Alle Goldstandard-Spans in eine flache Liste überführen
all_gold = [
    gold
    for golds in gold_by_start.values()
    for gold in golds
]

for pred_start, pred_end in predicted_spans:

    available = [
        gold for gold in all_gold
        if gold["task"] not in matched_tasks
    ]

    # 1. Perfect Match
    candidates = [
        gold for gold in available
        if gold["start"] == pred_start
        and gold["end"] == pred_end
    ]

    if candidates:
        gold = candidates[0]
        match_type = "perfect_match"

    else:
        # 2. Start Match
        candidates = [
            gold for gold in available
            if gold["start"] == pred_start
        ]

        if candidates:
            gold = candidates[0]
            match_type = "start_match"

        else:
            # 3. End Match
            candidates = [
                gold for gold in available
                if gold["end"] == pred_end
            ]

            if candidates:
                gold = candidates[0]
                match_type = "end_match"

            else:
                continue

    matches.append({
        "task": gold["task"],
        "pred_span": [pred_start, pred_end],
        "gold_span": [gold["start"], gold["end"]],
        "match_type": match_type
    })

    matched_tasks.add(gold["task"])

df_matches = pd.DataFrame(matches)

print("N matches: ", len(matches))
print(matches)


N matches:  4264
[{'task': '3903', 'pred_span': [17, 188], 'gold_span': [17, 188], 'match_type': 'perfect_match'}, {'task': '1794', 'pred_span': [190, 509], 'gold_span': [190, 509], 'match_type': 'perfect_match'}, {'task': '4575', 'pred_span': [511, 605], 'gold_span': [511, 746], 'match_type': 'start_match'}, {'task': '2349', 'pred_span': [748, 874], 'gold_span': [748, 874], 'match_type': 'perfect_match'}, {'task': '4788', 'pred_span': [876, 927], 'gold_span': [876, 980], 'match_type': 'start_match'}, {'task': '2260', 'pred_span': [982, 1107], 'gold_span': [982, 1107], 'match_type': 'perfect_match'}, {'task': '3430', 'pred_span': [1109, 1242], 'gold_span': [1109, 1242], 'match_type': 'perfect_match'}, {'task': '2311', 'pred_span': [1352, 1551], 'gold_span': [1352, 1551], 'match_type': 'perfect_match'}, {'task': '2276', 'pred_span': [1568, 1594], 'gold_span': [1568, 1594], 'match_type': 'perfect_match'}, {'task': '2417', 'pred_span': [1891, 1974], 'gold_span': [1885, 1974], 'match_type'

In [644]:
#prüfen
display(df_matches[df_matches["match_type"]=="end_match"]) 

,task,pred_span,gold_span,match_type
9,2417,"[1891, 1974]","[1885, 1974]",end_match
13,3830,"[2350, 2397]","[2340, 2397]",end_match
15,4898,"[2399, 2536]","[2454, 2536]",end_match
24,2822,"[3302, 3310]","[3293, 3310]",end_match
26,1162,"[3398, 3428]","[3374, 3428]",end_match
...,...,...,...,...
4233,420,"[591032, 591080]","[591054, 591080]",end_match
4250,5994,"[591750, 591838]","[591807, 591838]",end_match
4253,6229,"[591964, 592041]","[592028, 592041]",end_match
4256,5370,"[592202, 592206]","[592177, 592206]",end_match


### Nur vergleichbare Annotationen behalten: Task-IDs speichern

In [645]:

#taskid_pred_span_map = dict(zip(df_matches['task'], df_matches['pred_span'])) #key is task id und value die jeweilige span, value is list
#taskid_gold_span_map = dict(zip(df_matches['task'], df_matches['gold_span']))
relevant_task_ids = df_matches["task"].astype(int).tolist()
print(len(relevant_task_ids))


4264


In [646]:
print("Matches insgesamt:", len(matches))
print("Eindeutige Tasks in matches:", len(set(m["task"] for m in matches)))
print("Zeilen im DataFrame:", len(df_matches))
print("Eindeutige Tasks im DataFrame:", df_matches["task"].nunique())

Matches insgesamt: 4264
Eindeutige Tasks in matches: 4264
Zeilen im DataFrame: 4264
Eindeutige Tasks im DataFrame: 4264


In [647]:
display(all_gold_labels.head())

,Unnamed: 0,title,item,task,worker,label,ds_groundtruth,mace_ground_truth,glad_ground_truth
0,18692,Effi Briest,dem sich unerklärlicherweise Freund Alonzo anv...,1,gold_standard,stative_event,process,process,process
1,18695,Effi Briest,die sie noch immer geschlossen hielt,2,gold_standard,process,process,process,stative_event
2,18698,Effi Briest,"ja, Pfefferküchler Michelsen habe sogar gesagt",3,gold_standard,process,process,process,process
3,18701,Effi Briest,Aber ich habe mit ihm eine Partie zu Oberförst...,4,gold_standard,process,process,process,process
4,18704,Effi Briest,Da war irgendwo Krieg,5,gold_standard,stative_event,stative_event,stative_event,stative_event


### Golddataframe, der lediglich, die vergleichbaren items enthält

In [648]:
df_gold_comparison = all_gold_labels[all_gold_labels["task"].astype(int).isin(relevant_task_ids)].copy()

In [649]:
print(len(df_gold_comparison))
display(df_gold_comparison.head())

4264


,Unnamed: 0,title,item,task,worker,label,ds_groundtruth,mace_ground_truth,glad_ground_truth
0,18692,Effi Briest,dem sich unerklärlicherweise Freund Alonzo anv...,1,gold_standard,stative_event,process,process,process
1,18695,Effi Briest,die sie noch immer geschlossen hielt,2,gold_standard,process,process,process,stative_event
2,18698,Effi Briest,"ja, Pfefferküchler Michelsen habe sogar gesagt",3,gold_standard,process,process,process,process
4,18704,Effi Briest,Da war irgendwo Krieg,5,gold_standard,stative_event,stative_event,stative_event,stative_event
9,18719,Effi Briest,wie er Sie wegen des Kindes tot machen will,10,gold_standard,stative_event,process,process,process


#### Dictionary mit Task-ID und prediction erstellen:

In [650]:
span_predicted_map = {
    (anno["start"], anno["end"]): anno["predicted"]
    for anno in predictions["annotations"]
}

In [651]:
#basierend auf dem mapping die labels für die predicted spalte hinzufügen
df_matches["predicted"] = df_matches["pred_span"].apply(
    lambda span: span_predicted_map.get(tuple(span))
)

In [652]:
display(df_matches.head())

,task,pred_span,gold_span,match_type,predicted
0,3903,"[17, 188]","[17, 188]",perfect_match,stative_event
1,1794,"[190, 509]","[190, 509]",perfect_match,stative_event
2,4575,"[511, 605]","[511, 746]",start_match,process
3,2349,"[748, 874]","[748, 874]",perfect_match,stative_event
4,4788,"[876, 927]","[876, 980]",start_match,stative_event


In [653]:
# task id und das label mappen
taskid_predicted_map = dict(zip(
    df_matches["task"],
    df_matches["predicted"]
))

#task_id-label-mapping nutzen um die spalte in den gold_dataframe hinzuzufügen
df_gold_comparison["predicted"] = (
    df_gold_comparison["task"].astype(str).map(taskid_predicted_map)
)

In [654]:
df_gold_comparison[
    ["task", "label", "predicted","ds_groundtruth"]
].head(10)

,task,label,predicted,ds_groundtruth
0,1,stative_event,non_event,process
1,2,process,process,process
2,3,process,process,process
4,5,stative_event,stative_event,stative_event
9,10,stative_event,stative_event,process
10,11,stative_event,stative_event,stative_event
11,12,stative_event,stative_event,process
12,13,process,stative_event,process
13,14,process,non_event,non_event
14,15,process,process,process


In [655]:
df_gold_comparison["predicted"].isna().sum()

np.int64(0)

In [656]:
df_gold_comparison.head()

,Unnamed: 0,title,item,task,worker,label,ds_groundtruth,mace_ground_truth,glad_ground_truth,predicted
0,18692,Effi Briest,dem sich unerklärlicherweise Freund Alonzo anv...,1,gold_standard,stative_event,process,process,process,non_event
1,18695,Effi Briest,die sie noch immer geschlossen hielt,2,gold_standard,process,process,process,stative_event,process
2,18698,Effi Briest,"ja, Pfefferküchler Michelsen habe sogar gesagt",3,gold_standard,process,process,process,process,process
4,18704,Effi Briest,Da war irgendwo Krieg,5,gold_standard,stative_event,stative_event,stative_event,stative_event,stative_event
9,18719,Effi Briest,wie er Sie wegen des Kindes tot machen will,10,gold_standard,stative_event,process,process,process,stative_event


In [ ]:
#df_gold_comparison.to_csv("../data/krambambuli/261117_classifier_label_vgl.csv",index=False)

## 4. Jetzt Konfusionsmatrizen

In [442]:
from itertools import combinations

In [657]:

cols = ["label", "ds_groundtruth", "mace_ground_truth","glad_ground_truth","predicted"]

for col1, col2 in combinations(cols, 2):
    print(f"\n{col1} vs. {col2}")
    display(pd.crosstab(df_gold_comparison[col1], df_gold_comparison[col2]))


label vs. ds_groundtruth


ds_groundtruth,change_of_state,non_event,process,stative_event
label,,,,
change_of_state,1,2,23,4
non_event,0,1465,7,3
process,0,82,1470,2
stative_event,0,106,365,734



label vs. mace_ground_truth


mace_ground_truth,change_of_state,non_event,process,stative_event
label,,,,
change_of_state,2,2,22,4
non_event,0,1169,200,106
process,0,2,1548,4
stative_event,0,68,369,768



label vs. glad_ground_truth


glad_ground_truth,change_of_state,non_event,process,stative_event
label,,,,
change_of_state,4,2,13,11
non_event,0,1206,162,107
process,8,35,1360,151
stative_event,0,68,360,777


... (10 outputs left out)


mace_ground_truth vs. predicted


predicted,change_of_state,non_event,process,stative_event
mace_ground_truth,,,,
change_of_state,2,0,0,0
non_event,4,935,113,189
process,54,177,1484,424
stative_event,6,71,35,770



glad_ground_truth vs. predicted


predicted,change_of_state,non_event,process,stative_event
glad_ground_truth,,,,
change_of_state,7,0,5,0
non_event,4,960,154,193
process,41,148,1341,365
stative_event,14,75,132,825


## 5. F1-Scores

In [346]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    log_loss
)

In [506]:
models = {
    "Dawid & Skene": "ds_groundtruth",
    "MACE": "mace_ground_truth",
    "GLAD": "glad_ground_truth",
    "Manuell Gold": "label",
}

y_true = df_gold_comparison["predicted"]

results = []

for model_name, column in models.items():

    y_pred = df_gold_comparison[column]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall (macro)": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "F1 (macro)": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "F1 (weighted)": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
    })

results_df = pd.DataFrame(results)

display(results_df.round(3))

,Model,Accuracy,Precision (macro),Recall (macro),F1 (macro),F1 (weighted)
0,Dawid & Skene,0.817,0.570,0.597,0.579,0.796
1,MACE,0.828,0.705,0.625,0.625,0.813
2,GLAD,0.830,0.782,0.679,0.700,0.824
3,Manuell Gold,0.817,0.768,0.679,0.670,0.814


### Pro Label

In [ ]:
reports = []

for model_name, column in models.items():

    report = classification_report(
        df_gold_comparison["predicted"],
        df_gold_comparison[column],
        zero_division=0,
        output_dict=True
    )

    for category, metrics in report.items():

        if isinstance(metrics, dict):
            reports.append({
                "Model": model_name,
                "Category": category,
                "Precision": metrics["precision"],
                "Recall": metrics["recall"],
                "F1-Score": metrics["f1-score"],
                "Support": metrics["support"]
            })

report_df = pd.DataFrame(reports)



In [508]:
display(report_df)

,Model,Category,Precision,Recall,F1-Score,Support
0,Dawid & Skene,change_of_state,0.000000,0.000000,0.000000,27.0
1,Dawid & Skene,non_event,0.672566,0.873563,0.760000,87.0
2,Dawid & Skene,process,0.873684,0.904632,0.888889,367.0
3,Dawid & Skene,stative_event,0.733333,0.611111,0.666667,72.0
4,Dawid & Skene,macro avg,0.569896,0.597327,0.578889,553.0
5,Dawid & Skene,weighted avg,0.781113,0.817360,0.796279,553.0
6,MACE,change_of_state,0.500000,0.074074,0.129032,27.0
7,MACE,non_event,0.741573,0.758621,0.750000,87.0
8,MACE,process,0.877285,0.915531,0.896000,367.0
9,MACE,stative_event,0.701299,0.750000,0.724832,72.0


In [ ]:
#report_df.to_csv("../data/krambambuli/261116_predicted_labelscores.csv",index=False, float_format="%.3f",decimal=",")
